# AI-Based Customer Segmentation & Personalized Marketing Recommendation System
### AIML Mini-Project: Unsupervised Machine Learning, RFM Analysis & Automated Persona Profiling
---
**Core Objectives:**
1. **Exploratory Data Analysis (EDA):** Understand customer distributions across purchasing indicators.
2. **Feature Engineering & RFM Modeling:** Compute Recency, Frequency, and Monetary value metrics.
3. **Optimal Cluster Selection:** Evaluate Within-Cluster Sum of Squares (Elbow Method) and Silhouette Coefficients.
4. **Unsupervised Clustering:** Train K-Means and compare with Hierarchical Agglomerative Clustering.
5. **Automated Persona Profiling:** Derive business names (Champions, At-Risk, Savers) from centroid statistics.
6. **Prescriptive Marketing Recommendations:** Tailor actionable campaigns for each segment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

sns.set_theme(style="whitegrid")
print("Libraries successfully imported!")

## 1. Dataset Loading and Inspection
We load both the **Online Retail (RFM)** dataset and the classic **Mall Customers** dataset.

In [ ]:
# Load Online Retail RFM Dataset
df_rfm = pd.read_csv("../data/online_retail_rfm.csv")
print("--- Online Retail RFM Dataset (Top 5 rows) ---")
display(df_rfm.head())
print(df_rfm.info())

# Load Mall Customers Dataset
df_mall = pd.read_csv("../data/mall_customers.csv")
print("\n--- Mall Customers Dataset (Top 5 rows) ---")
display(df_mall.head())

## 2. Exploratory Data Analysis (EDA)
Examine distributions of key customer features: Recency, Frequency, and Monetary spend.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df_rfm['Recency'], kde=True, ax=axes[0], color='royalblue')
axes[0].set_title("Recency (Days Since Last Order)")

sns.histplot(df_rfm['Frequency'], kde=True, ax=axes[1], color='forestgreen')
axes[1].set_title("Frequency (Total Orders)")

sns.histplot(df_rfm['Monetary'], kde=True, ax=axes[2], color='darkorange')
axes[2].set_title("Monetary ($ Total Spend)")

plt.tight_layout()
plt.show()

## 3. Feature Scaling
Because K-Means calculates Euclidean distance, features on larger numeric scales (e.g., Monetary in thousands) would dominate features with smaller scales (e.g., Frequency). Hence, standardization ($z = \frac{x - \mu}{\sigma}$) is essential.

In [ ]:
features = ['Recency', 'Frequency', 'Monetary']
X = df_rfm[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Scaled Feature Matrix Shape: {X_scaled.shape}")

## 4. Finding Optimal Clusters: Elbow Method & Silhouette Analysis
- **Elbow Method:** Plots Inertia (Within-Cluster Sum of Squares, WCSS) vs $k$.
- **Silhouette Analysis:** Measures inter-cluster separation vs intra-cluster cohesion.

In [ ]:
k_range = range(2, 9)
wcss = []
sil_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    wcss.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.plot(k_range, wcss, marker='o', color='crimson', linewidth=2)
ax1.set_title("Elbow Method (WCSS vs k)", fontsize=13)
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("Inertia (WCSS)")

ax2.bar(k_range, sil_scores, color='teal', alpha=0.85)
ax2.set_title("Silhouette Scores by k", fontsize=13)
ax2.set_xlabel("Number of Clusters (k)")
ax2.set_ylabel("Silhouette Coefficient")

plt.tight_layout()
plt.show()

best_k = list(k_range)[np.argmax(sil_scores)]
print(f"Optimal k based on peak Silhouette Score: k = {best_k} (Score: {max(sil_scores):.4f})")

## 5. Model Training: K-Means Clustering
We fit K-Means with the chosen cluster count ($k=4$) and assign cluster labels to each customer.

In [ ]:
k_final = 4
kmeans_model = KMeans(n_clusters=k_final, init='k-means++', n_init=10, random_state=42)
df_rfm['Cluster'] = kmeans_model.fit_predict(X_scaled)

print(f"Model Convergence Inertia: {kmeans_model.inertia_:.2f}")
print(f"Final Silhouette Score: {silhouette_score(X_scaled, df_rfm['Cluster']):.4f}")
print(f"Davies-Bouldin Index: {davies_bouldin_score(X_scaled, df_rfm['Cluster']):.4f}")

## 6. Cluster Profiling & Automated Segment Naming
We calculate the centroids (mean values) of Recency, Frequency, and Monetary spend to understand the customer segments.

In [ ]:
cluster_summary = df_rfm.groupby('Cluster').agg(
    CustomerCount=('CustomerID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean'),
    Total_Revenue=('Monetary', 'sum')
).reset_index()

# Intelligent Business Labeling
def label_segment(row, global_means):
    if row['Avg_Recency'] < global_means['Recency'] and row['Avg_Monetary'] >= global_means['Monetary']:
        return "Champions & VIP High Spenders"
    elif row['Avg_Recency'] < global_means['Recency'] and row['Avg_Frequency'] >= global_means['Frequency']:
        return "Loyal Frequent Shoppers"
    elif row['Avg_Recency'] >= global_means['Recency'] and row['Avg_Monetary'] >= global_means['Monetary']:
        return "At-Risk Lapsed Spenders"
    else:
        return "Hibernating / Budget Shoppers"

global_means = df_rfm[['Recency', 'Frequency', 'Monetary']].mean()
cluster_summary['Persona_Name'] = cluster_summary.apply(lambda r: label_segment(r, global_means), axis=1)
cluster_summary['Revenue_Share_%'] = round((cluster_summary['Total_Revenue'] / df_rfm['Monetary'].sum()) * 100, 1)

display(cluster_summary)

## 7. Cluster Visualization
Plotting Recency vs Monetary and Frequency vs Monetary with distinct cluster coloring.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

palette = sns.color_palette("Set2", k_final)
sns.scatterplot(data=df_rfm, x='Recency', y='Monetary', hue='Cluster', palette=palette, s=50, alpha=0.8, ax=ax1)
ax1.set_title("Customer Clusters: Recency vs Monetary Spend", fontsize=13)

sns.scatterplot(data=df_rfm, x='Frequency', y='Monetary', hue='Cluster', palette=palette, s=50, alpha=0.8, ax=ax2)
ax2.set_title("Customer Clusters: Frequency vs Monetary Spend", fontsize=13)

plt.tight_layout()
plt.show()

## 8. Personalized Marketing Strategy Recommendations
| Customer Segment | Core Characteristics | Marketing Objective | Recommended Campaign & Offer |
| :--- | :--- | :--- | :--- |
| **Champions & VIPs** | High spend, frequent, recent visits | VIP Retention & High Touch | Exclusive VIP Previews, Zero-threshold perks, concierge service |
| **Loyal Frequent Shoppers** | Consistent orders, moderate spend | Basket Size Expansion | Tiered discounts ($20 off on orders > $120), bundle incentives |
| **At-Risk Lapsed Spenders** | High lifetime spend, declining activity | Urgent Win-Back & Retention | Comeback voucher ($25 cart credit), customer feedback survey |
| **Hibernating / Budget** | Infrequent, low spend, high days lapsed | Low-Cost Reactivation | Automated seasonal clearance blasts, bulk markdown deals |